In [ ]:
pip install lime scikit-image matplotlib

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet50
model = models.resnet50(weights=None)

# Change final layer
num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Correct Kaggle path
model_path = "/kaggle/input/datasets/munigetiganesh/resnet/resnet50_xray_best (1).pth"
# Load weights
model.load_state_dict(torch.load(model_path, map_location=device))

model = model.to(device)
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
def predict_fn(images):

    model.eval()

    batch = []

    for img in images:
        img = Image.fromarray(img.astype('uint8'), 'RGB')
        img = transform(img)
        batch.append(img)

    batch = torch.stack(batch).to(device)

    with torch.no_grad():
        outputs = model(batch)
        probs = torch.nn.functional.softmax(outputs, dim=1)

    return probs.cpu().numpy()

In [ ]:
explainer = lime_image.LimeImageExplainer()

explanation = explainer.explain_instance(
    img_np,
    predict_fn,
    top_labels=2,
    hide_color=0,
    num_samples=1000
)

In [ ]:
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=5,
    hide_rest=False
)

plt.figure(figsize=(6,6))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation")
plt.axis("off")
plt.show()